# 서울시 119 구급·구조 활동 데이터 전처리

## 0. 환경 설정

In [1]:
import pandas as pd
import numpy as np
import re
import os
import glob
import unicodedata

SRC_DIR = "C:/ai/source/09_Project/data"
OUT_DIR = "C:/ai/source/09_Project/data/outputs"
os.makedirs(OUT_DIR, exist_ok=True)

pd.set_option('display.max_columns', 20)

# 모델 학습용 통합 데이터를 저장할 팀 공용 폴더 (루트 경로)
ML_DATA_DIR = "C:/ai/source/09_Project/ml_data"

try:
    os.makedirs(ML_DATA_DIR, exist_ok=True)
except OSError:
    print(f"{ML_DATA_DIR} 폴더를 만들 수 없어 {OUT_DIR} 에 저장합니다.")
    ML_DATA_DIR = OUT_DIR


## 1. 소방서 → 자치구 매핑 준비

In [2]:
station_df = pd.read_csv(f"{SRC_DIR}/서울소방서 119안전센터 현황.csv", encoding="cp949")

def extract_gu(text):
    if pd.isna(text):
        return None
    m = re.search(r'([가-힣]+구)', str(text))
    return m.group(1) if m else None

station_df['District_guess'] = station_df['관할구역'].apply(extract_gu)
STATION2GU = station_df.groupby('서별')['District_guess'] \
    .agg(lambda x: x.value_counts().index[0]).to_dict()

VALID_GU = {'종로구','중구','용산구','성동구','광진구','동대문구','중랑구','성북구','강북구','도봉구',
            '노원구','은평구','서대문구','마포구','양천구','강서구','구로구','금천구','영등포구',
            '동작구','관악구','서초구','강남구','송파구','강동구'}

print(f"매핑된 소방서 수: {len(STATION2GU)} / 표준 자치구 수: {len(VALID_GU)}")
STATION2GU


매핑된 소방서 수: 25 / 표준 자치구 수: 25


{'강남소방서': '강남구',
 '강동소방서': '강동구',
 '강북소방서': '강북구',
 '강서소방서': '강서구',
 '관악소방서': '관악구',
 '광진소방서': '광진구',
 '구로소방서': '구로구',
 '금천소방서': '금천구',
 '노원소방서': '노원구',
 '도봉소방서': '도봉구',
 '동대문소방서': '동대문구',
 '동작소방서': '동작구',
 '마포소방서': '마포구',
 '서대문소방서': '서대문구',
 '서초소방서': '서초구',
 '성동소방서': '성동구',
 '성북소방서': '성북구',
 '송파소방서': '송파구',
 '양천소방서': '양천구',
 '영등포소방서': '영등포구',
 '용산소방서': '용산구',
 '은평소방서': '은평구',
 '종로소방서': '종로구',
 '중랑소방서': '중랑구',
 '중부소방서': '중구'}

## 2. 구급활동 실적 (Prophet용) — `ambulance_prophet.csv`

`119_구급활동_실적_소방서별__2026...csv` (2005~2025년, 소방서 단위)는 3개 헤더 행(연도 / 대분류 / 소분류)이 컬럼에 걸쳐 있습니다.
- Row0(컬럼명) : 연도 (예: `2005`, `2005.1`, ...)
- Row1(iloc[0]) : 대분류 (출동건수(건), 이송건수(건), 이송인원(명) ...)
- Row2(iloc[1]) : 소분류 (소계, 급만성질환, 교통사고 ...)
- Row3(iloc[2:]) : 실제 데이터 (소방서별 값)

In [3]:
SRC = f"{SRC_DIR}/119+구급활동+실적(소방서별)_20260921151015.csv"

df = pd.read_csv(SRC, encoding="utf-8")
level1 = df.iloc[0]
level2 = df.iloc[1]
data = df.iloc[2:].reset_index(drop=True)
cols = df.columns.tolist()
years = [c.split('.')[0] for c in cols]
id0, id1 = cols[0], cols[1]

records = []
for i, col in enumerate(cols):
    if col in (id0, id1) or not years[i].isdigit():
        continue
    yr = years[i]
    l1 = str(level1[col]).strip()
    l2 = str(level2[col]).strip()
    
    if l2 != '소계':
        continue
    if l1.startswith('출동건수'):
        metric = 'y'
    elif l1.startswith('이송건수'):
        metric = 'transport_cnt'
    elif l1.startswith('이송인원'):
        metric = 'transport_person'
    else:
        continue
        
    for _, row in data.iterrows():
        station = row[id1]
        if station not in STATION2GU:
            continue
            
        raw = row[col]
        is_missing = (str(raw).strip() == '-') or pd.isna(raw)
        try:
            val = float(str(raw).replace(',', '')) if not is_missing else 0.0
        except Exception:
            val = 0.0
            is_missing = True
            
        records.append({
            'ds': f'{yr}-01-01', 'district': STATION2GU[station],
            'metric': metric, 'value': val, 'imputed_zero': is_missing
        })

long_df = pd.DataFrame(records)
wide = long_df.pivot_table(index=['ds','district'], columns='metric',
                            values='value', aggfunc='first').reset_index()
flag = long_df[long_df['metric']=='y'][['ds','district','imputed_zero']]
ambulance_prophet = wide.merge(flag, on=['ds','district'], how='left')
ambulance_prophet = ambulance_prophet[['ds','district','y','transport_cnt','transport_person','imputed_zero']]
ambulance_prophet = ambulance_prophet.sort_values(['district','ds']).reset_index(drop=True)

print(ambulance_prophet.shape)
print("imputed_zero(0으로 대체된 결측) 건수:", ambulance_prophet['imputed_zero'].sum())
ambulance_prophet.to_csv(f"{OUT_DIR}/ambulance_prophet.csv", index=False, encoding='utf-8-sig')
ambulance_prophet.head()


(525, 6)
imputed_zero(0으로 대체된 결측) 건수: 37


,ds,district,y,transport_cnt,transport_person,imputed_zero
0,2005-01-01,강남구,16577.0,10367.0,10639.0,False
1,2006-01-01,강남구,16905.0,10701.0,10911.0,False
2,2007-01-01,강남구,18384.0,11474.0,11731.0,False
3,2008-01-01,강남구,19178.0,11723.0,12021.0,False
4,2009-01-01,강남구,20152.0,12223.0,12521.0,False


## 3. 서울생활인구 (Random Forest용) — `population_rf.csv`

In [4]:
SRC = f"{SRC_DIR}/자치구단위 서울생활인구 일별 집계표.csv"
pop = pd.read_csv(SRC, encoding="cp949")

pop['ds'] = pd.to_datetime(pop['기준일ID'].astype(str), format='%Y%m%d')
pop = pop[pop['시군구명'] != '서울시'].copy()   # 자치구 단위만 (서울시 합계행 제외)
pop = pop.rename(columns={
    '시군구명': 'district',
    '총생활인구수': 'total_pop',
    '주간인구수(09~18)': 'day_pop',
    '야간인구수(19~08)': 'night_pop',
})

keep_cols = ['ds','district','total_pop','내국인생활인구수','장기체류외국인인구수','단기체류외국인인구수',
             '일최대인구수','일최소인구수','day_pop','night_pop','일최대이동인구수',
             '서울외유입인구수','동일자치구행정동간이동인구수','자치구간이동인구수']
pop = pop[keep_cols]

full_dates = pd.date_range(pop['ds'].min(), pop['ds'].max(), freq='D')

filled = []
for gu, g in pop.groupby('district'):
    g = g.set_index('ds').sort_index().reindex(full_dates)
    g['district'] = gu
    
    num_cols = [c for c in keep_cols if c not in ('ds','district')]
    g[num_cols] = g[num_cols].interpolate(method='linear', limit_direction='both') # 선형 보간
    
    g.index.name = 'ds'
    filled.append(g.reset_index())

population_rf = pd.concat(filled, ignore_index=True)
population_rf['pop_peak'] = population_rf['일최대인구수']
population_rf['pop_diff'] = population_rf['day_pop'] - population_rf['night_pop']
population_rf = population_rf.sort_values(['district','ds']).reset_index(drop=True)

print(population_rf.shape)
population_rf.to_csv(f"{OUT_DIR}/population_rf.csv", index=False, encoding='utf-8')
population_rf.head()

(76000, 16)


,ds,district,total_pop,내국인생활인구수,장기체류외국인인구수,단기체류외국인인구수,일최대인구수,일최소인구수,day_pop,night_pop,일최대이동인구수,서울외유입인구수,동일자치구행정동간이동인구수,자치구간이동인구수,pop_peak,pop_diff
0,2018-04-05,강남구,912863.8679,871371.5215,22101.3558,19390.9906,1.147911e+06,692878.7272,1.105394e+06,775342.1904,751766.5021,226143.0841,140222.3607,385401.0573,1.147911e+06,330052.0261
1,2018-04-06,강남구,917979.5577,873013.6363,22909.3283,22056.5931,1.151434e+06,700679.3649,1.109721e+06,781021.4959,753218.8548,227849.4783,139374.3635,385995.0130,1.151434e+06,328699.3484
2,2018-04-07,강남구,794301.8799,751015.8833,19614.8350,23671.1616,9.206919e+05,698966.3936,8.837356e+05,730420.6343,508315.4071,160249.2826,126454.2611,221611.8634,9.206919e+05,153314.9895
3,2018-04-08,강남구,737387.8758,699746.8916,17194.4820,20446.5022,8.256871e+05,677309.1636,7.948478e+05,696345.0471,418042.9103,120661.7740,126125.1247,171256.0116,8.256871e+05,98502.7887
4,2018-04-09,강남구,894796.3158,853966.4341,21776.1550,19053.7266,1.127112e+06,676373.4418,1.088129e+06,756701.7096,741235.0435,220709.6647,138905.9884,381619.3904,1.127112e+06,331427.0548


## 4. 사고 원인 통계 — `accident_cause_long.csv` / `accident_cause_onehot.csv`

In [5]:
SRC = f"{SRC_DIR}/119+구조활동+실적(구별)_20260921153018.csv"
df = pd.read_csv(SRC, encoding="utf-8")

lvl1, lvl2, lvl3 = df.iloc[0], df.iloc[1], df.iloc[2]
data = df.iloc[3:].reset_index(drop=True)
cols = df.columns.tolist()
years = [c.split('.')[0] for c in cols]
id0, id1 = cols[0], cols[1]

records = []
for i, col in enumerate(cols):
    if col in (id0, id1) or not years[i].isdigit():
        continue
    yr = years[i]
    if str(lvl1[col]).strip() != '구조인원 (명)' or str(lvl2[col]).strip() != '사고종별구조인원':
        continue
    cause = str(lvl3[col]).strip()
    if cause == '소계':
        continue
    for _, row in data.iterrows():
        gu = row[id1]
        if gu not in VALID_GU:
            continue
        raw = row[col]
        missing = (str(raw).strip() in ('-', 'nan')) or pd.isna(raw)
        try:
            val = float(str(raw).replace(',', '')) if not missing else 0.0
        except Exception:
            val = 0.0
        records.append({'ds': f'{yr}-01-01', 'district': gu, 'cause': cause, 'rescued_count': val})

cause_long = pd.DataFrame(records)

# 원인명 표준화 (연도별 지표 구성 차이 통합) + 결측/미정의 카테고리 -> 기타/미상
CAUSE_MAP = {
    '화재':'화재','교통사고':'교통사고','수난사고':'수난사고','기계사고':'기계사고',
    '건물사고':'건물사고','승강기사고':'승강기사고','산악사고':'산악사고','인명갇힘':'인명갇힘',
    '폭발사고':'폭발사고','잠금장치 개방':'잠금장치개방','안전조치':'안전조치',
    '자살추정':'자살추정','위치추적':'위치추적','기타':'기타/미상',
}
cause_long['cause'] = cause_long['cause'].map(CAUSE_MAP).fillna('기타/미상')

accident_cause_long = cause_long.groupby(['ds','district','cause'], as_index=False)['rescued_count'].sum()
accident_cause_long.to_csv(f"{OUT_DIR}/accident_cause_long.csv", index=False, encoding='utf-8')

# One-Hot Encoding (구·연도 단위로 원인별 건수를 컬럼화)
accident_cause_onehot = accident_cause_long.pivot_table(
    index=['ds','district'], columns='cause', values='rescued_count',
    aggfunc='sum', fill_value=0).reset_index()
accident_cause_onehot.to_csv(f"{OUT_DIR}/accident_cause_onehot.csv", index=False, encoding='utf-8')
accident_cause_long.head()


,ds,district,cause,rescued_count
0,2006-01-01,강남구,건물사고,378.0
1,2006-01-01,강남구,교통사고,134.0
2,2006-01-01,강남구,기계사고,11.0
3,2006-01-01,강남구,산악사고,153.0
4,2006-01-01,강남구,수난사고,19.0


## 5. 보조 데이터 3종

1. **`ambulance_monthly_gu_2022_2024.csv`**
2. **`ambulance_citywide_monthly.csv`**
3. **`ambulance_sample_gwanak_202506.csv`**

In [6]:
# 5-1) 소방청 전국 데이터 -> 서울 전체 월별 보조 시계열
d2020 = pd.read_csv(f"{SRC_DIR}/소방청_본부별 구급활동정보_20201231.csv", encoding='cp949')
d2023 = pd.read_csv(f"{SRC_DIR}/소방청_본부별 구급활동정보_20231231.csv", encoding='cp949')

nat = pd.concat([d2020, d2023], ignore_index=True).drop_duplicates(subset=['년도','월','본부구분'])
seoul_city = nat[nat['본부구분'] == '서울'].copy()
seoul_city['ds'] = pd.to_datetime(seoul_city['년도'].astype(str) + '-' +
                                   seoul_city['월'].astype(str).str.zfill(2) + '-01')

seoul_city = seoul_city.rename(columns={'출동건수':'y','이송건수':'transport_cnt','이송환자수':'transport_person'})
seoul_city['District'] = '서울시_전체(구분류불가)'
ambulance_citywide_monthly = seoul_city[['ds','District','y','transport_cnt','transport_person']].sort_values('ds')
ambulance_citywide_monthly.to_csv(f"{OUT_DIR}/ambulance_citywide_monthly.csv", index=False, encoding='utf-8-sig')
print("서울 전체 월별:", ambulance_citywide_monthly.shape)

# 5-2) 서울시 소방 구급 출동 현황(xlsx) -> 구단위 월별 (2022-2024)
xl = pd.ExcelFile(f"{SRC_DIR}/서울시 소방 구급 출동 현황(2022_2024).xlsx")

frames = []
for sheet in xl.sheet_names:
    year = re.search(r'\d{4}', sheet).group()
    d = xl.parse(sheet)
    d = d[d['구분'].isin(VALID_GU)].copy()
    d = d.groupby('구분', as_index=False).sum(numeric_only=True)
    long_ = d.melt(id_vars='구분', var_name='월', value_name='y')
    long_['월'] = long_['월'].str.replace('월', '').astype(int)
    long_['ds'] = pd.to_datetime(year + '-' + long_['월'].astype(str).str.zfill(2) + '-01')
    long_ = long_.rename(columns={'구분': 'District'})[['ds', 'District', 'y']]
    frames.append(long_)

ambulance_monthly_gu = pd.concat(frames, ignore_index=True).sort_values(['District', 'ds']).reset_index(drop=True)
ambulance_monthly_gu['y'] = ambulance_monthly_gu['y'].fillna(0)
ambulance_monthly_gu.to_csv(f"{OUT_DIR}/ambulance_monthly_gu_2022_2024.csv", index=False, encoding='utf-8-sig')
print("구단위 월별(2022-2024):", ambulance_monthly_gu.shape)

# 5-3) data.csv(관악소방서 샘플) -> 참고용 단일 포인트
sample = pd.read_csv(f"{SRC_DIR}/data.csv", encoding='utf-8')
ambulance_sample_gwanak = pd.DataFrame([{
    'ds': '2025-06-01', 'District': '관악구', 'y': len(sample),
    'note': 'data.csv 원본 행수 기준 집계 (API 샘플 추정, 전수조사 여부 불확실 - 검증 필요)'
}])
ambulance_sample_gwanak.to_csv(f"{OUT_DIR}/ambulance_sample_gwanak_202506.csv", index=False, encoding='utf-8-sig')
print("관악구 샘플 포인트:", len(sample), "건")

서울 전체 월별: (48, 5)
구단위 월별(2022-2024): (900, 3)
관악구 샘플 포인트: 1000 건


## 6. 구급출동 건별 데이터 (2021~2022) — `ambulance_incident_monthly.csv`

In [7]:
INCIDENT_COLS = ['DCLR_YMD', 'GRNDS_SGG_NM', 'PTN_OCRN_TYPE_NM', 'TRANS_CLSF_NM']

paths = sorted(glob.glob(f"{SRC_DIR}/서울소방재난본부_구급출동 현황_*.csv"))
if len(paths) == 0:
    raise FileNotFoundError(f"{SRC_DIR} 에서 서울소방재난본부_구급출동 현황_*.csv 를 찾지 못했습니다.")

frames = []
for p in paths:
    part = pd.read_csv(p, encoding="utf-8", usecols=INCIDENT_COLS, low_memory=False)
    print(f"{os.path.basename(p)} : {len(part):,}건")
    frames.append(part)
incident = pd.concat(frames, ignore_index=True)

# 날짜 변환 : 정수 -> 문자열 -> 날짜형, 그 다음 월 첫날로 통일
incident['ds'] = pd.to_datetime(incident['DCLR_YMD'].astype(str), format='%Y%m%d')
incident['ds'] = incident['ds'].dt.to_period('M').dt.to_timestamp()
incident = incident.rename(columns={'GRNDS_SGG_NM': 'district'})

# 서울 25개 자치구만 남기기
before = len(incident)
incident = incident[incident['district'].isin(VALID_GU)].copy()
print(f"서울 25개 자치구 외 제외: {before - len(incident):,}건 (남은 건수 {len(incident):,}건)")

# 조건별 True/False 열 -> 합계가 곧 건수
incident['cnt'] = 1
incident['transport'] = incident['TRANS_CLSF_NM'].isin(['이송', '연계이송'])
incident['no_transport'] = incident['TRANS_CLSF_NM'] == '미이송'
incident['fire_activity'] = incident['TRANS_CLSF_NM'] == '소방활동'
incident['disease'] = incident['PTN_OCRN_TYPE_NM'] == '질병'
incident['non_disease'] = incident['PTN_OCRN_TYPE_NM'] == '질병외'
incident['etc'] = incident['PTN_OCRN_TYPE_NM'] == '기타'

ambulance_incident_monthly = incident.groupby(['ds', 'district'], as_index=False).agg(
    y=('cnt', 'sum'),
    transport_cnt=('transport', 'sum'),
    no_transport_cnt=('no_transport', 'sum'),
    fire_activity_cnt=('fire_activity', 'sum'),
    disease_cnt=('disease', 'sum'),
    non_disease_cnt=('non_disease', 'sum'),
    etc_cnt=('etc', 'sum'),
)
ambulance_incident_monthly = ambulance_incident_monthly.sort_values(['district', 'ds']).reset_index(drop=True)

# 검산 : 출동건수 = 이송 + 미이송 + 소방활동
diff = (ambulance_incident_monthly['y'] - ambulance_incident_monthly['transport_cnt']
        - ambulance_incident_monthly['no_transport_cnt'] - ambulance_incident_monthly['fire_activity_cnt']).abs().sum()
print("이송구분 합계와 출동건수 차이:", diff)

# 5절 xlsx(2022)와 연간 합계 비교 (소폭 차이는 집계 기준 차이로 추정)
inc_2022 = ambulance_incident_monthly[ambulance_incident_monthly['ds'].dt.year == 2022]['y'].sum()
xlsx_2022 = ambulance_monthly_gu[ambulance_monthly_gu['ds'].dt.year == 2022]['y'].sum()
print(f"2022년 합계 - 건별 데이터: {inc_2022:,.0f} / xlsx: {xlsx_2022:,.0f}")

ambulance_incident_monthly.to_csv(f"{OUT_DIR}/ambulance_incident_monthly.csv", index=False, encoding='utf-8-sig')
print(ambulance_incident_monthly.shape)
ambulance_incident_monthly.head()

서울소방재난본부_구급출동 현황_2021.csv : 551,597건
서울소방재난본부_구급출동 현황_2022.csv : 617,414건
서울 25개 자치구 외 제외: 1,694건 (남은 건수 1,167,317건)
이송구분 합계와 출동건수 차이: 0
2022년 합계 - 건별 데이터: 617,256 / xlsx: 620,693
(600, 9)


,ds,district,y,transport_cnt,no_transport_cnt,fire_activity_cnt,disease_cnt,non_disease_cnt,etc_cnt
0,2021-01-01,강남구,2327,1307,777,243,870,1370,87
1,2021-02-01,강남구,2173,1191,753,229,791,1294,88
2,2021-03-01,강남구,2595,1373,967,255,902,1560,133
3,2021-04-01,강남구,2646,1473,919,254,960,1554,132
4,2021-05-01,강남구,2929,1637,937,355,1121,1710,98


## 7. 의료이용 유출입 데이터 (2019~2024)

1. **수요(거주지 기준)** : 그 구에 사는 사람이 어디서 입원했는지 → `유출률`
2. **공급(병원 소재지 기준)** : 그 구의 병원이 어디 환자를 받았는지 → `유입률`
3. **OD 매트릭스** : 서울 구 ↔ 구 이동

In [8]:
MEDFLOW_YEARS = [2019, 2020, 2021, 2022, 2023, 2024]
TOTAL_COL = "입원 전체_입원 전체"
ER_COL = "진료분야별_응급실"

def find_medflow_file(year):
    for name in sorted(os.listdir(SRC_DIR)):
        norm = unicodedata.normalize("NFC", name)
        if norm.startswith(f"{year}년") and norm.lower().endswith(".xlsx"):
            return f"{SRC_DIR}/{name}"
    raise FileNotFoundError(
        f"{SRC_DIR} 폴더에 '{year}년' xlsx 파일이 없습니다. "
    )

def parse_medflow_xlsx(path, year):
    """3행 헤더를 하나의 컬럼명으로 합쳐 읽고, 서울 관련 행만 남긴 뒤 숫자로 변환"""
    raw = pd.read_excel(path, header=None)
    row1 = raw.iloc[1].ffill()   # 중분류
    row2 = raw.iloc[2]           # 소분류
    data = raw.iloc[3:].reset_index(drop=True)

    cols = []
    for i in range(raw.shape[1]):
        if i < 8:
            side = "환자" if i < 4 else "병원"
            field = ["코드", "시도", "중진료권", "시군구"][i % 4]
            cols.append(f"{side}_{field}")
        else:
            cols.append(f"{row1.iloc[i]}_{row2.iloc[i]}")
    data.columns = cols

    # 끝에 붙은 완전히 빈 열 제거
    data = data.loc[:, ~data.columns.str.fullmatch("nan_nan")]
    data.insert(0, "연도", year)

    # 환자 거주지 또는 병원 소재지가 서울인 행만
    mask = (data["환자_시도"] == "서울특별시") | (data["병원_시도"] == "서울특별시")
    data = data[mask].reset_index(drop=True)

    id_cols = ["연도", "환자_코드", "환자_시도", "환자_중진료권", "환자_시군구",
               "병원_코드", "병원_시도", "병원_중진료권", "병원_시군구"]
    value_cols = [c for c in data.columns if c not in id_cols]
    # 숫자로 변환
    data[value_cols] = data[value_cols].apply(
        lambda s: pd.to_numeric(s.astype(str).str.replace(",", "", regex=False).str.strip(), errors="coerce")
    )
    return data

medflow_full = pd.concat(
    [parse_medflow_xlsx(find_medflow_file(year), year) for year in MEDFLOW_YEARS],
    ignore_index=True,
)
print("의료이용 유출입(서울 관련):", medflow_full.shape)

# 7-1) 수요 : 서울 거주자 기준
demand = medflow_full[medflow_full["환자_시도"] == "서울특별시"].copy()
demand["서울내치료"] = demand["병원_시도"] == "서울특별시"

medical_demand = demand.groupby(["연도", "환자_시군구"], as_index=False).agg(
    총입원연인원=(TOTAL_COL, "sum"),
    서울내입원연인원=(TOTAL_COL, lambda s: s[demand.loc[s.index, "서울내치료"]].sum()),
)
medical_demand["서울외유출연인원"] = medical_demand["총입원연인원"] - medical_demand["서울내입원연인원"]
medical_demand["유출률"] = (medical_demand["서울외유출연인원"] / medical_demand["총입원연인원"]).round(4)
medical_demand = medical_demand.rename(columns={"환자_시군구": "district"})

er_demand = demand.groupby(["연도", "환자_시군구"], as_index=False)[ER_COL].sum(min_count=1) \
    .rename(columns={"환자_시군구": "district", ER_COL: "응급실입원연인원"})
medical_demand = medical_demand.merge(er_demand, on=["연도", "district"], how="left")

# 7-2) 공급 : 서울 병원 소재지 기준
supply = medflow_full[medflow_full["병원_시도"] == "서울특별시"].copy()
supply["서울내환자"] = supply["환자_시도"] == "서울특별시"

medical_supply = supply.groupby(["연도", "병원_시군구"], as_index=False).agg(
    총처리연인원=(TOTAL_COL, "sum"),
    서울내환자연인원=(TOTAL_COL, lambda s: s[supply.loc[s.index, "서울내환자"]].sum()),
)
medical_supply["서울외유입연인원"] = medical_supply["총처리연인원"] - medical_supply["서울내환자연인원"]
medical_supply["유입률"] = (medical_supply["서울외유입연인원"] / medical_supply["총처리연인원"]).round(4)
medical_supply = medical_supply.rename(columns={"병원_시군구": "district"})

er_supply = supply.groupby(["연도", "병원_시군구"], as_index=False)[ER_COL].sum(min_count=1) \
    .rename(columns={"병원_시군구": "district", ER_COL: "응급실처리연인원"})
medical_supply = medical_supply.merge(er_supply, on=["연도", "district"], how="left")

# 7-3) OD 매트릭스 : 서울 -> 서울
seoul_to_seoul = medflow_full[(medflow_full["환자_시도"] == "서울특별시") &
                               (medflow_full["병원_시도"] == "서울특별시")]
medical_od_matrix = seoul_to_seoul.groupby(["연도", "환자_시군구", "병원_시군구"], as_index=False)[TOTAL_COL].sum()
medical_od_matrix = medical_od_matrix.rename(columns={
    "환자_시군구": "환자_district", "병원_시군구": "병원_district", TOTAL_COL: "입원연인원"})

# 연도를 다른 표와 같은 ds(ISO 날짜) 형식으로 통일 (연 단위 데이터는 매년 1월 1일로 표기)
for df_ in (medical_demand, medical_supply, medical_od_matrix):
    df_.insert(0, "ds", pd.to_datetime(df_["연도"].astype(str) + "-01-01"))
    df_.drop(columns="연도", inplace=True)

medical_demand.to_csv(f"{OUT_DIR}/medical_flow_district_demand.csv", index=False, encoding="utf-8-sig")
medical_supply.to_csv(f"{OUT_DIR}/medical_flow_district_supply.csv", index=False, encoding="utf-8-sig")
medical_od_matrix.to_csv(f"{OUT_DIR}/medical_flow_seoul_od_matrix.csv", index=False, encoding="utf-8-sig")

print("수요:", medical_demand.shape, " 공급:", medical_supply.shape, " OD:", medical_od_matrix.shape)
medical_demand.head()

의료이용 유출입(서울 관련): (71250, 43)
수요: (150, 7)  공급: (150, 7)  OD: (3750, 4)


,ds,district,총입원연인원,서울내입원연인원,서울외유출연인원,유출률,응급실입원연인원
0,2019-01-01,강남구,1002129,582232,419897,0.4190,NaN
1,2019-01-01,강동구,915412,637381,278031,0.3037,NaN
2,2019-01-01,강북구,910537,692635,217902,0.2393,NaN
3,2019-01-01,강서구,1419243,895605,523638,0.3690,NaN
4,2019-01-01,관악구,1141886,804299,337587,0.2956,NaN


## 8. 모델 학습용 통합 데이터 — `monthly_gu_integrated.csv`

In [10]:
# 생활인구 : 일별 -> 월평균
pop_monthly = population_rf.copy()
pop_monthly['ds'] = pop_monthly['ds'].dt.to_period('M').dt.to_timestamp()
pop_cols = ['total_pop', 'day_pop', 'night_pop', 'pop_peak', 'pop_diff']
pop_monthly = pop_monthly.groupby(['ds', 'district'], as_index=False)[pop_cols].mean()
pop_monthly[pop_cols] = pop_monthly[pop_cols].round(0)

integrated = ambulance_incident_monthly.merge(pop_monthly, on=['ds', 'district'], how='left')

# 의료이용 유출입 : 연 단위 지표를 연도 + 자치구 기준으로 붙이기
integrated['연도'] = integrated['ds'].dt.year
demand_year = medical_demand.assign(연도=medical_demand['ds'].dt.year)[['연도', 'district', '유출률', '응급실입원연인원']]
supply_year = medical_supply.assign(연도=medical_supply['ds'].dt.year)[['연도', 'district', '유입률', '응급실처리연인원']]
integrated = integrated.merge(demand_year, on=['연도', 'district'], how='left')
integrated = integrated.merge(supply_year, on=['연도', 'district'], how='left')
integrated = integrated.drop(columns='연도')

print("통합 데이터:", integrated.shape)
print("결측치가 있는 행 수:", len(integrated[integrated.isna().any(axis=1)]))

integrated.to_csv(f"{ML_DATA_DIR}/monthly_gu_integrated.csv", index=False, encoding="utf-8")
integrated.head()

통합 데이터: (600, 18)
결측치가 있는 행 수: 0


,ds,district,y,transport_cnt,no_transport_cnt,fire_activity_cnt,disease_cnt,non_disease_cnt,etc_cnt,total_pop,day_pop,night_pop,pop_peak,pop_diff,유출률,응급실입원연인원,유입률,응급실처리연인원
0,2021-01-01,강남구,2327,1307,777,243,870,1370,87,793528.0,924939.0,699663.0,961287.0,225276.0,0.3913,89369.0,0.6068,233772.0
1,2021-02-01,강남구,2173,1191,753,229,791,1294,88,801462.0,939367.0,702958.0,978505.0,236409.0,0.3913,89369.0,0.6068,233772.0
2,2021-03-01,강남구,2595,1373,967,255,902,1560,133,836926.0,992624.0,725714.0,1032963.0,266909.0,0.3913,89369.0,0.6068,233772.0
3,2021-04-01,강남구,2646,1473,919,254,960,1554,132,828750.0,984379.0,717587.0,1024186.0,266792.0,0.3913,89369.0,0.6068,233772.0
4,2021-05-01,강남구,2929,1637,937,355,1121,1710,98,809312.0,949535.0,709153.0,986772.0,240382.0,0.3913,89369.0,0.6068,233772.0


## 9. 결과 요약

In [ ]:
summary = {
    'ambulance_prophet.csv': ambulance_prophet.shape,
    'population_rf.csv': population_rf.shape,
    'accident_cause_long.csv': accident_cause_long.shape,
    'accident_cause_onehot.csv': accident_cause_onehot.shape,
    'ambulance_citywide_monthly.csv': ambulance_citywide_monthly.shape,
    'ambulance_monthly_gu_2022_2024.csv': ambulance_monthly_gu.shape,
    'ambulance_sample_gwanak_202506.csv': ambulance_sample_gwanak.shape,
    'ambulance_incident_monthly.csv': ambulance_incident_monthly.shape,
    'medical_flow_district_demand.csv': medical_demand.shape,
    'medical_flow_district_supply.csv': medical_supply.shape,
    'medical_flow_seoul_od_matrix.csv': medical_od_matrix.shape,
    'monthly_gu_integrated.csv (ML_DATA_DIR)': integrated.shape,
}
for k, v in summary.items():
    print(f"{k:45s} {v}")